In [ ]:
import pyspark.sql.functions as f
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import regexp_extract
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.ml.feature import OneHotEncoder, StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler, StandardScaler

In [ ]:
import findspark
findspark.init()

import pyspark
conf = (
    pyspark.SparkConf()
        .setMaster('local[*]')
)
sc = pyspark.SparkContext(conf=conf)
sc

In [ ]:
from pyspark.sql import SQLContext
spark = SQLContext(sc)

from pyspark.sql.types import StringType, ShortType, IntegerType, DoubleType, StructType, StructField, BooleanType
df_schema = StructType([
                        StructField('property_type',StringType()),
                        StructField('price',StringType()),
                        StructField('location',StringType()),
                        StructField('city',StringType()),
                        StructField('bath',StringType()),
                        StructField('purpose',StringType()),
                        StructField('bedrooms',StringType()),
                        StructField('area_in_marla',StringType())
                        ])

df = spark.read.csv("house_prices.csv",schema=df_schema, header = True)

df.show(5)


## Check shape of data 

In [ ]:
num_rows = df.count()

num_columns = len(df.columns)

# Print the shape
print("Shape of DataFrame: ({}, {})".format(num_rows, num_columns))

In [ ]:
#df = df.withColumn("id", regexp_extract(df["property_type"], r'(\d+)', 1).cast(StringType()))

# Showing the modified DataFrame
df.show(20)

### Change data type of data

In [ ]:
for col_name in df.columns:
    if col_name == "property_type":
        df = df.withColumn('property_type', regexp_replace(df['property_type'], '[0-9"",]', ''))
    else:
        df = df.withColumn(col_name, regexp_replace(df[col_name], '"', ''))

df.show(5)

In [ ]:
df.printSchema()

In [ ]:
columns_to_convert = ['price', 'bath', 'bedrooms', 'area_in_marla']

for col in columns_to_convert:
    df = df.withColumn(col, df[col].cast(IntegerType()))
    
df = df.drop("location")
    
df.show(5)

## Check missing value 

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum

missing_values = df.select(*(spark_sum(col(c).isNull().cast('int')).alias(c) for c in df.columns))
missing_values.show()

In [ ]:
df = df.dropna()

## Check Inconsistent data

In [ ]:
string_columns = ['property_type', 'city', 'purpose']

for col_name in string_columns:
    unique_values = df.select(col_name).distinct().collect()
    print("Unique values for column '{}':".format(col_name))
    for value in unique_values:
        print(value[col_name])
    print("\n")

In [ ]:
df = df.filter(col("price") > 0)

## Encoding Data 

In [ ]:
df.show(5)

In [ ]:
categorical_columns = ['property_type', 'city', 'purpose']

indexers = [StringIndexer(inputCol=column, outputCol=column+"_index").fit(df) for column in categorical_columns]
indexed_df = df
for indexer in indexers:
    indexed_df = indexer.transform(indexed_df)


encoder = OneHotEncoder(inputCols=[column+"_index" for column in categorical_columns],
                        outputCols=[column+"_encoded" for column in categorical_columns])
encoded_df = encoder.fit(indexed_df).transform(indexed_df)

encoded_df = encoded_df.drop(*[column+"_index" for column in categorical_columns])
df_encoded = encoded_df.select("property_type_encoded", "city_encoded", "purpose_encoded", "price", "bath", "bedrooms", "area_in_marla")


In [ ]:
df_encoded.show()

## Feature Visualization

In [ ]:
df_pd = df_encoded.toPandas()

numerical_columns = ['price', 'bath', 'bedrooms', 'area_in_marla']
num_rows = (len(numerical_columns) + 1) // 2
num_cols = 2
fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 10))
fig.subplots_adjust(hspace=0, wspace=0)

if num_rows > 1:
    axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    ax = axes[i]
    sns.histplot(df_pd[col], bins=20, color='lightblue', alpha=0.7, ax=ax)
    hist, bins = np.histogram(df_pd[col], bins=10, density=True)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    ax.plot(bin_centers, hist, color='red', linestyle='--')
    ax.set_title(f'{col.capitalize()} Histogram')
    ax.set_xlabel(col.capitalize())
    ax.set_ylabel('Density')

plt.tight_layout()
plt.show()

In [ ]:
from pyspark.sql.functions import col, when

def handle_outliers(df, columns, action='remove'):
    print("Shape before cleansing:", (df.count(), len(df.columns)))

    result_df = df
    
    for column in columns:
        df.select(column).toPandas().boxplot()
        plt.title(f'Box plot of {column} before handling outliers')
        plt.show()

    for column in columns:
        quantiles = df.approxQuantile(column, [0.25, 0.75], 0.05)
        q1 = quantiles[0]
        q3 = quantiles[1]
        iqr = q3 - q1

        # Define bounds
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        # Count outliers
        outliers_count = df.filter((col(column) < lower_bound) | (col(column) > upper_bound)).count()
        print(f"Number of outliers in column '{column}': {outliers_count}")

        if action == 'remove':
            result_df = result_df.filter((col(column) >= lower_bound) & (col(column) <= upper_bound))
        elif action == 'cap':
            result_df = result_df.withColumn(column, when(col(column) < lower_bound, lower_bound).when(col(column) > upper_bound, upper_bound).otherwise(col(column)))
        elif action == 'pass':
            pass
        else:
            raise ValueError("Invalid action. Supported actions: 'remove', 'cap_lower', 'cap_upper', 'pass'")

    print("Shape after cleansing:", (result_df.count(), len(result_df.columns)))

    return result_df


columns_to_handle = ['price', 'area_in_marla']
action_to_take = 'remove'  
df_handled = handle_outliers(df_encoded, columns_to_handle, action=action_to_take)


In [ ]:
df_handled.show()

In [ ]:
assembler = VectorAssembler(
    inputCols=["property_type_encoded", "city_encoded", "purpose_encoded", "bath", "bedrooms", "area_in_marla"],
    outputCol="features")

df_assembled = assembler.transform(df_handled)
df_assembled.show()

In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True, withMean=False)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

In [ ]:
df_scaled.show()

In [ ]:
final_data = df_scaled.select("scaled_features", "price")
final_data.show(5)

## Train Test Split

In [ ]:
train_data, test_data = final_data .randomSplit([0.7, 0.3], seed=42)

## linear regression, Visualization

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
import matplotlib.pyplot as plt
import numpy as np
lr = LinearRegression(featuresCol="scaled_features", labelCol="price")
lr_model = lr.fit(train_data)
predictions_train = lr_model.transform(train_data)
predictions_test = lr_model.transform(test_data)

# Evaluation
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction")

train_mae = evaluator.evaluate(predictions_train, {evaluator.metricName: "mae"})
test_mae = evaluator.evaluate(predictions_test, {evaluator.metricName: "mae"})

train_rmse = evaluator.evaluate(predictions_train, {evaluator.metricName: "rmse"})
test_rmse = evaluator.evaluate(predictions_test, {evaluator.metricName: "rmse"})

train_mse = evaluator.evaluate(predictions_train, {evaluator.metricName: "mse"})
test_mse = evaluator.evaluate(predictions_test, {evaluator.metricName: "mse"})

# Visualization
predictions_train_pd = predictions_train.select("price", "prediction").toPandas()
predictions_test_pd = predictions_test.select("price", "prediction").toPandas()

plt.figure(figsize=(18, 6))

# Plot for train data
plt.subplot(1, 2, 1)
plt.scatter(predictions_train_pd["prediction"], predictions_train_pd["price"], color='blue', label='Actual')
plt.plot(predictions_train_pd["prediction"], predictions_train_pd["prediction"], color='red', label='Predicted') 
plt.title("Train Data: Predicted vs. Actual")
plt.xlabel("Predicted Price")
plt.ylabel("Actual Price")
plt.grid(True)
plt.legend()

# Plot for test data
plt.subplot(1, 2, 2)
plt.scatter(predictions_test_pd["prediction"], predictions_test_pd["price"], color='blue', label='Actual')
plt.plot(predictions_test_pd["prediction"], predictions_test_pd["prediction"], color='red', label='Predicted') 
plt.title("Test Data: Predicted vs. Actual")
plt.xlabel("Predicted Price")
plt.ylabel("Actual Price")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

##  Evaluation Model

In [ ]:
print("Train MAE:", train_mae)
print("Test MAE:", test_mae)
print("Train RMSE:", train_rmse)
print("Test RMSE:", test_rmse)
print("Train MSE:", train_mse)
print("Test MSE:", test_mse)
